# 02: ADMET gauntlet

**The strategic question:** is our data an advantage, or are we reproducing what is already open?

This notebook runs every featurizer against every split on every endpoint and prints one table. The table is the deliverable, not a model and not a leaderboard position.

Real public data: ESOL solubility (Delaney) and three Tox21 assays.

In [ ]:
import os, sys
from pathlib import Path
# make the notebook runnable from anywhere
HERE = Path.cwd()
if not (HERE / "run.py").exists():
    HERE = Path(globals().get("__vsc_ipynb_file__", ".")).resolve().parent
os.chdir(HERE)
sys.path.insert(0, str(HERE))
print("working in:", Path.cwd())

In [ ]:
from src.data import download, load_endpoint
from src.featurize import available

download()
for name in ["esol", "tox21:NR-AR", "tox21:SR-MMP", "tox21:NR-AhR"]:
    print(" ", load_endpoint(name).describe())

print("\nfeaturizer availability:")
for k, v in available().items():
    print(f"  {k:<12} {v}")

## 1. Splits are the whole ballgame

| Split | What it simulates |
|---|---|
| `random` | interpolating within chemistry you have already explored. Optimistic. |
| `scaffold` | Bemis-Murcko. Test set has chemotypes the model has not seen. |
| `temporal` | next quarter's compounds. **Needs a run-date column.** |

Public benchmark sets have no assay dates, so `temporal` cannot run on them. That is not a limitation of this code. It is the single most important thing to fix in your own data export, and the error message says so.

In [ ]:
from src.data import load_endpoint
from src.featurize import parse
from src.splits import temporal_split

ep = load_endpoint("esol")
mols, keep = parse(ep.smiles)
try:
    temporal_split(ep.date)
except ValueError as e:
    print(e)

## 2. What a scaffold split actually does

It groups molecules by their Bemis-Murcko core and puts the rare cores in test, where they belong.

In [ ]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import Counter

scaffolds = Counter(MurckoScaffold.MurckoScaffoldSmiles(mol=m) for m in mols)
print(f"{len(mols)} molecules -> {len(scaffolds)} distinct scaffolds")
print(f"singleton scaffolds: {sum(1 for v in scaffolds.values() if v == 1)}")
print("\nlargest scaffold groups:")
for s, n in scaffolds.most_common(4):
    print(f"  {n:>4}  {s or '(acyclic)'}")

## 3. Run the gauntlet

Three arms that need only RDKit. `chemberta` joins automatically once `torch` and `transformers` are installed.

In [ ]:
from src.gauntlet import run, report

df = run(endpoints=["esol", "tox21:NR-AR", "tox21:SR-MMP", "tox21:NR-AhR"],
         featurizers=["morgan", "descriptors", "combo"],
         splits=["random", "scaffold"])
df.head(12)

In [ ]:
verdicts = report(df, baseline="morgan")

## 4. Read the two findings

**The random split inflates everything.** On ESOL, RMSE goes from 0.58 to 0.93 when the test set contains scaffolds the model has not seen. On SR-MMP, ROC-AUC drops from 0.93 to 0.84. Scaffold splitting costs something in all twelve endpoint-arm combinations, between 0.018 and 0.108 ROC-AUC and 48% to 60% added RMSE on ESOL. That gap is the share of your reported accuracy that is memorisation of chemotypes. Any ADMET number quoted without naming its split is unreadable.

**But do not over-read the ranking.** A single-seed run of this notebook shows the winning arm changing with the split on three of four endpoints. Repeat it across seeds with bootstrap intervals (section 7) and that collapses to one of four: `tox21:NR-AR`.

That one is worth looking at. NR-AR is also the only endpoint where all three arms sit inside each other's bootstrap intervals, in both splits. On ESOL and SR-MMP, where the descriptors are cleanly separated from the fingerprint, the winner does not change at all. So the ranking moves exactly where nothing can be told apart, which is a better argument for the gate than the flip ever was.

A 2026 systematic benchmark of molecular property prediction reports that method rankings are unstable across evaluation protocols, with four different models taking the top score across six ADME endpoints under temporal splitting. What this notebook supports is narrower: on the one endpoint here where the arms are indistinguishable, the split decides the ranking.

The practical consequence is not "foundation models are bad". It is: **run the gate per endpoint, ship per endpoint, and prefer the cheaper arm when the intervals overlap.** There is no one model.

In [ ]:
import pandas as pd

# the shift, made explicit
piv = df.pivot_table(index=["endpoint", "featurizer"], columns="split", values="primary")
piv["shift"] = (piv["scaffold"] - piv["random"]).round(4)
piv.round(4)

## 5. The gate

An arm ships for an endpoint only if it beats the `morgan` fingerprint baseline under the hardest available split, independently, on that endpoint.

If nothing beats the fingerprint, ship the fingerprint. It is faster and interpretable, and you can defend the choice.

In [ ]:
import json
print(json.dumps(verdicts, indent=2))

## 7. Now put error bars on it

Everything above is a single seed. Before quoting any of it, repeat across seeds and bootstrap the test set, and let the headline claim fail if it deserves to.

In [ ]:
from src.experiments import run as run_experiments

# ESOL only, so the notebook stays quick. The full grid across all four
# endpoints is `python run.py experiments`: a few minutes, same code.
exp = run_experiments(endpoints=["esol"],
                      featurizers=["morgan", "descriptors", "combo"],
                      splits=["random", "scaffold"],
                      seeds=3, n_boot=200)

On ESOL the split shift survives: the random and scaffold intervals do not overlap on any arm.

The *ranking between arms* does not: `descriptors` and `combo` sit inside each other's intervals. Reporting one as the winner would be reporting noise.

Run the full grid (`python run.py experiments`) and the same holds on all four endpoints, which is what withdrew the 3-of-4 claim.

Results are written to `results/with_intervals.csv` and `results/claim_check.json` so every number in the write-up is traceable to a file rather than to a screenshot.

## 6. Your own assay data

```
python run.py internal exports/logD_2019_2026.csv \
    --smiles-col canonical_smiles --y-col logD --date-col assay_date \
    --task regression --splits random scaffold temporal
```

Three columns needed: SMILES, the measured value, and **the assay run date**. The third is the one usually missing and the one that matters. Without it you cannot run a temporal split, and without a temporal split your accuracy is the accuracy of interpolating within chemistry you already explored.

One thing worth doing before the first model: check that the same compound tested twice under the same protocol agrees with itself. That reproducibility figure is the ceiling on anything trained on the data, and it is the only honest answer to "how good could this get?"

In [ ]:
from src.data import load_internal
help(load_internal)